# k-NN, Naive Bayes and SVM — Solution

**Dataset:** Titanic passengers; numerical inputs for k-NN/SVM and sex/class for Naive Bayes.

Run from top to bottom. The steps match the exercise; short answers explain the results.

## Libraries and settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score

## 1. Compute one neighbour distance

Replace the gap with `np.sqrt(np.sum((new_point - known_point) ** 2))`. If 3 of 5 neighbours are positive, what is the class probability?

In [ ]:
new_point = np.array([2, 3])
known_point = np.array([5, 7])
distance = np.sqrt(np.sum((new_point - known_point) ** 2))
print('Distance:', distance)
print('Positive probability:', 3 / 5)

**Answer:** The distance is 5 and the class probability is 0.6. The majority class is positive.

## 2. Prepare the data

Preparation is provided. Which inputs have different units? Why must the target be excluded from input distances?

In [ ]:
titanic = pd.read_csv('../00_Data/titanic.csv')
# Fixed, simple preparation: use rows with known age and fare.
# Removing missing rows may change which passengers are represented.
titanic = titanic.dropna(subset=['Age', 'Fare']).copy()
X = titanic[['Age', 'Fare', 'Pclass']].copy()
X['male'] = (titanic['Sex'] == 'male').astype(int)
y = titanic['Survived']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.head())
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

**Answer:** Age is in years, fare in pounds, and class/sex are coded categories. Survival is the answer; including it would reveal the answer to the model.

## 3. Train a scaled k-NN

Replace the gap with `KNeighborsClassifier(n_neighbors=5)`. The scaler is provided inside a pipeline. Run CV before fitting and final test evaluation.

In [ ]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
print('Scaled k-NN CV accuracy:', round(scores.mean(), 3))
knn.fit(X_train, y_train)
print('Fixed k=5 test accuracy:', round(knn.score(X_test, y_test), 3))
new_passenger = pd.DataFrame({'Age': [30], 'Fare': [80], 'Pclass': [1], 'male': [0]})
print('P(survived):', knn.predict_proba(new_passenger)[:, 1])

**Answer:** The pipeline learns scaling within each CV fold. The final fitted model uses training scaling for the test data and the new passenger. The survival probability is based on the five neighbours.

## 4. See scaling and the choice of k

The comparison is supplied. Compare scaled and unscaled CV accuracy, then inspect only three k values. Do not use test scores to choose k.

In [ ]:
unscaled = KNeighborsClassifier(n_neighbors=5)
print('Unscaled CV accuracy:', round(cross_val_score(
    unscaled, X_train, y_train, cv=5, scoring='accuracy').mean(), 3))
for k in [3, 5, 15]:
    candidate = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    score = cross_val_score(candidate, X_train, y_train, cv=5, scoring='accuracy').mean()
    print('k:', k, 'scaled CV accuracy:', round(score, 3))

**Answer:** Compare the validation scores rather than assuming scaling always raises accuracy on every dataset. Scaling changes distances. CV can guide k; the fixed k=5 example above is not retrospectively changed using its test score.

## 5. Work a Naive Bayes example

Use the supplied scores from the golf example. Replace the gap with `no / (yes + no)`. Which class has the larger score?

In [ ]:
yes = (9 / 14) * (2 / 9) * (3 / 9) * (3 / 9) * (3 / 9)
no = (5 / 14) * (3 / 5) * (1 / 5) * (4 / 5) * (3 / 5)
p_no = no / (yes + no)
print('Scores yes/no:', round(yes, 4), round(no, 4))
print('P(no):', round(p_no, 3))

**Answer:** No has the larger score and normalised probability about 0.795. The multiplication follows the conditional-independence assumption.

## 6. Apply categorical Naive Bayes

The code is provided. Here passenger class and sex are categories, not continuous distances. Why does this model use fewer inputs than k-NN?

In [ ]:
cat_train = X_train[['male', 'Pclass']].copy()
cat_test = X_test[['male', 'Pclass']].copy()
cat_train['Pclass'] = cat_train['Pclass'] - 1
cat_test['Pclass'] = cat_test['Pclass'] - 1
nb = CategoricalNB(min_categories=[2, 3])
nb.fit(cat_train, y_train)
print('Naive Bayes test accuracy:', round(nb.score(cat_test, y_test), 3))
woman_first_class = pd.DataFrame({'male': [0], 'Pclass': [0]})
print('P(survived):', nb.predict_proba(woman_first_class)[:, 1].round(3))

**Answer:** CategoricalNB models category frequencies. We use two clearly categorical inputs to keep the example simple. Age and fare would need another modelling choice, such as binning; these test scores are not a comparison of identical feature sets.

## 7. Train a simple SVM

Replace the gap with `SVC(kernel="linear", C=1)`. The fixed model is evaluated once; the two extra kernel examples use training CV only. Which kernel allows a curved boundary?

In [ ]:
svm = make_pipeline(StandardScaler(), SVC(kernel="linear", C=1))
svm.fit(X_train, y_train)
print('Fixed linear SVM test accuracy:', round(svm.score(X_test, y_test), 3))
print('Number of support vectors:', svm[-1].n_support_.sum())
for kernel in ['linear', 'rbf']:
    candidate = make_pipeline(StandardScaler(), SVC(kernel=kernel, C=1))
    scores = cross_val_score(candidate, X_train, y_train, cv=5, scoring='accuracy')
    print(kernel, 'CV accuracy:', round(scores.mean(), 3))

**Answer:** RBF permits curved boundaries; linear SVM uses a straight boundary in scaled input space. Support vectors help determine the solution. Compare CV results for kernel choices; no single kernel is always best.

### Jupyter notebook — footer info

Include this information when submitting your notebook.

In [ ]:
import platform
from datetime import datetime
print(platform.platform())
print("Python:", platform.python_version())
print("Date:", datetime.now().isoformat(timespec="seconds"))